# Portable TRM GEMM Refiner

This notebook is the main entrypoint for running the Turing-first TRM GEMM prototype.

Recommended runtime:
- Google Colab
- GPU: `T4`

What it does:
- installs the package
- checks backend/runtime mode
- runs the Colab smoke test
- generates offline traces
- builds the tiny recursive model
- runs a short rollout


In [1]:
# If running in Colab, mount or clone the repo first, then set the project root below.
import os
from pathlib import Path

PROJECT_ROOT = Path('/content/youtube-videos')
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path.cwd()

print('PROJECT_ROOT =', PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

PROJECT_ROOT = /content


In [2]:
!pip install -e .
!pip install -r requirements-colab.txt

Obtaining file:///content
ERROR: file:///content does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements-colab.txt'


In [ ]:
!python -m trm_gemm.colab_smoke

In [ ]:
import torch

from trm_gemm.backend import TritonGemmBackend
from trm_gemm.baselines import greedy_search, random_search
from trm_gemm.data import TraceDataset, TraceGenerationConfig, generate_trace_records
from trm_gemm.model import TinyRecursiveGemmRefiner, rollout_refiner
from trm_gemm.training import compute_losses
from trm_gemm.types import GemmTaskSpec, T4

backend = TritonGemmBackend()
print({
    'backend_mode': backend.mode,
    'cuda_available': torch.cuda.is_available(),
    'device_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'triton_available': backend.capabilities.triton_available,
})

In [ ]:
tasks = [
    GemmTaskSpec(512, 512, 512),
    GemmTaskSpec(1024, 512, 768),
    GemmTaskSpec(1536, 1024, 640),
    GemmTaskSpec(2048, 1024, 1024),
]

records = generate_trace_records(
    tasks,
    T4,
    backend,
    TraceGenerationConfig(seeds_per_task=2, max_steps_per_seed=3),
)

dataset = TraceDataset(records)
print('num_records =', len(dataset))
print('first_record =', dataset[0].to_dict())

In [ ]:
model = TinyRecursiveGemmRefiner()
batch = records[: min(8, len(records))]
losses = compute_losses(model, batch)
print({k: round(v.item(), 4) for k, v in losses.items()})

In [ ]:
task = tasks[0]
random_schedule, random_feedback = random_search(backend, task, T4, budget=6, seed=0)
greedy_schedule, greedy_feedback = greedy_search(backend, task, T4, budget=6)

print('random_schedule =', random_schedule.to_dict())
print('random_feedback =', random_feedback.to_dict())
print('greedy_schedule =', greedy_schedule.to_dict())
print('greedy_feedback =', greedy_feedback.to_dict())

In [ ]:
edits = rollout_refiner(model, task, T4, random_schedule, random_feedback, steps=4)
print('rollout_edits =', [edit.to_dict() for edit in edits])

In [ ]:
dataset_path = PROJECT_ROOT / 'tmp_trm_gemm_traces.jsonl'
dataset.to_jsonl(dataset_path)
print('saved', dataset_path)